In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: Eine Qiskit-Function von Multiverse Computing
> **Note:** * Qiskit Functions sind eine experimentelle Funktion, die nur für Nutzer des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) verfügbar ist. Sie befinden sich im Preview-Status und können sich noch ändern.

## Überblick
Mit der Funktion „Singularity Machine Learning - Classification" kannst du reale Machine-Learning-Probleme auf Quantenhardware lösen, ohne Quantenexpertise zu benötigen. Diese Application Function basiert auf Ensemble-Methoden und ist ein hybrider Klassifikator. Sie nutzt klassische Methoden wie Boosting, Bagging und Stacking für das initiale Ensemble-Training. Anschließend werden Quantenalgorithmen wie der Variational Quantum Eigensolver (VQE) und der Quantum Approximate Optimization Algorithm (QAOA) eingesetzt, um die Diversität, Generalisierungsfähigkeiten und Gesamtkomplexität des trainierten Ensembles zu verbessern.

Im Gegensatz zu anderen Quantenmaschinenlernlösungen ist diese Funktion in der Lage, große Datensätze mit Millionen von Beispielen und Merkmalen zu verarbeiten, ohne durch die Anzahl der Qubits im Ziel-QPU eingeschränkt zu sein. Die Anzahl der Qubits bestimmt lediglich die Größe des Ensembles, das trainiert werden kann. Sie ist außerdem sehr flexibel und kann zur Lösung von Klassifikationsproblemen in einem breiten Anwendungsbereich eingesetzt werden, darunter Finanzen, Gesundheitswesen und Cybersicherheit.
Sie erzielt durchgehend hohe Genauigkeiten bei klassisch anspruchsvollen Problemen mit hochdimensionalen, verrauschten und unausgewogenen Datensätzen.
![Funktionsweise](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Sie wurde entwickelt für:
1. Ingenieure und Datenwissenschaftler in Unternehmen, die ihr Technologieangebot durch die Integration von Quantenmaschinenlernens in ihre Produkte und Dienstleistungen erweitern möchten,
2. Forscher in Quantenforschungslaboren, die Anwendungen des Quantenmaschinenlernens erkunden und Quantencomputing für Klassifikationsaufgaben nutzen möchten, und
3. Studierende und Lehrende an Bildungseinrichtungen in Kursen wie maschinelles Lernen, die die Vorteile des Quantencomputings demonstrieren möchten.

Das folgende Beispiel zeigt die verschiedenen Funktionalitäten, darunter `create`, `list`, `fit` und `predict`, und demonstriert deren Verwendung bei einem synthetischen Problem aus zwei ineinandergreifenden Halbkreisen – ein notorisch anspruchsvolles Problem aufgrund seiner nichtlinearen Entscheidungsgrenze.


## Funktionsbeschreibung
Diese Qiskit-Function ermöglicht es Nutzern, binäre Klassifikationsprobleme mit Singularitys quantengestütztem Ensemble-Klassifikator zu lösen. Im Hintergrund verwendet sie einen hybriden Ansatz, um ein Ensemble von Klassifikatoren auf dem gelabelten Datensatz klassisch zu trainieren und es anschließend mit dem Quantum Approximate Optimization Algorithm (QAOA) auf IBM&reg; QPUs für maximale Diversität und Generalisierung zu optimieren. Über eine benutzerfreundliche Schnittstelle können Nutzer einen Klassifikator gemäß ihren Anforderungen konfigurieren, ihn auf dem gewünschten Datensatz trainieren und damit Vorhersagen auf einem zuvor nicht gesehenen Datensatz treffen.

Um ein allgemeines Klassifikationsproblem zu lösen:
1. Verarbeite den Datensatz vor und teile ihn in Trainings- und Testmengen auf. Optional kannst du die Trainingsmenge weiter in Trainings- und Validierungsmengen aufteilen. Dies kann mit [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html) erreicht werden.
2. Wenn die Trainingsmenge unausgewogen ist, kannst du sie mit [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets) neu sampeln, um die Klassen auszubalancieren.
3. Lade die Trainings-, Validierungs- und Testmengen separat in den Speicher der Function mit der Methode `file_upload` des Katalogs hoch und übergib dabei jeweils den relevanten Pfad.
4. Initialisiere den Quantenklassifikator mit der Aktion `create` der Function, die Hyperparameter wie die Anzahl und Typen der Lerner, die Regularisierung (Lambda-Wert) und Optimierungsoptionen einschließlich der Anzahl der Schichten, den Typ des klassischen Optimierers, das Quantum Backend und so weiter akzeptiert.
5. Trainiere den Quantenklassifikator auf der Trainingsmenge mit der Aktion `fit` der Function, übergib die gelabelte Trainingsmenge und ggf. die Validierungsmenge.
6. Erstelle Vorhersagen auf der zuvor nicht gesehenen Testmenge mit der Aktion `predict` der Function.
## Aktionsbasierter Ansatz
Die Function verwendet einen aktionsbasierten Ansatz. Du kannst sie dir als virtuelle Umgebung vorstellen, in der du Aktionen verwendest, um Aufgaben auszuführen oder ihren Zustand zu ändern. Derzeit bietet sie folgende Aktionen: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) und [create_fit_predict](#7-create-fit-predict). Das folgende Beispiel demonstriert die Aktion `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

Die Aktion `list` ruft alle gespeicherten Klassifikatoren im Format `*.pkl.tar` aus dem gemeinsamen Datenverzeichnis ab. Du kannst auch auf den Inhalt dieses Verzeichnisses über die Methode `catalog.files()` zugreifen. Im Allgemeinen sucht die List-Aktion nach Dateien mit der Erweiterung `*.pkl.tar` im gemeinsamen Datenverzeichnis und gibt sie in einem Listenformat zurück.

#### Eingaben

|     Name    |    Typ      | Beschreibung |   Erforderlich  |
|-------------|-------------|--------------|-----------------|
| `action` | `str` | Der Name der Aktion aus `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` und `delete`. | Ja |

#### Verwendung

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
Die Aktion `create` erstellt einen Klassifikator des angegebenen Typs `quantum_classifier` anhand der bereitgestellten Parameter und speichert ihn im gemeinsamen Datenverzeichnis.

> **Note:** Die Function unterstützt derzeit nur den `QuantumEnhancedEnsembleClassifier`.
#### Eingaben
|     Name    |    Typ      | Beschreibung | Erforderlich  | Standard |
|-------------|-------------|--------------|---------------|---------|
| `action` | `str` | Der Name der Aktion aus `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` und `delete`. | Ja | - |
| `name` | `str` | Der Name des Quantenklassifikators, z.B. `spam_classifier`. | Ja | - |
| `instance` | `str` | IBM-Instanz. | Ja | - |
| `backend_name` | `str` | IBM-Rechenressource. Standard ist `None`, d.h. das Backend mit den wenigsten ausstehenden Jobs wird verwendet. | Nein | `None` |
| `quantum_classifier` | `str` | Der Typ des Quantenklassifikators, d.h. `QuantumEnhancedEnsembleClassifier`. | Nein | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Die Anzahl der Lerner im Ensemble. | Nein | `10` |
| `learners_types` | `list` | Typen der Lerner. Zu den unterstützten Typen gehören: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` und `LogisticRegression`. Weitere Details zu jedem Typ findest du in der [scikit-learn-Dokumentation](https://scikit-learn.org/stable/supervised_learning.html). | Nein | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Anteile der einzelnen Lernertypen im Ensemble. | Nein | `[1.0]` |
| `learners_options` | `list` | Optionen für jeden Lernertyp im Ensemble. Eine vollständige Liste der Optionen für den/die gewählten Lernertyp(en) findest du in der [scikit-learn-Dokumentation](https://scikit-learn.org/stable/supervised_learning.html). | Nein | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` oder `list` | Typ(en) der zu verwendenden Regularisierung: `onsite` oder `alpha`. `onsite` steuert den Onsite-Term, wobei höhere Werte zu spärlicheren Ensembles führen. `alpha` steuert den Kompromiss zwischen Interaktions- und Onsite-Term, wobei niedrigere Werte zu spärlicheren Ensembles führen. Wird eine Liste angegeben, werden Modelle für jeden Typ trainiert und das beste ausgewählt. | Nein | `onsite` |
| `regularization` | `str` oder `float` oder `list` | Regularisierungswert. Begrenzt auf `0` bis `+inf`, wenn `regularization_type` `onsite` ist. Begrenzt auf `0` bis `1`, wenn `regularization_type` `alpha` ist. Bei `auto` wird eine Auto-Regularisierung verwendet – der optimale Regularisierungsparameter wird per binärer Suche mit dem gewünschten Verhältnis ausgewählter zu gesamten Klassifikatoren (`regularization_desired_ratio`) und der oberen Schranke für den Regularisierungsparameter (`regularization_upper_bound`) ermittelt. Wird eine Liste angegeben, werden Modelle für jeden Wert trainiert und das beste ausgewählt. | Nein | `0.01` |
| `regularization_desired_ratio` | `float` oder `list` | Gewünschtes Verhältnis/Verhältnisse ausgewählter zu gesamten Klassifikatoren für die Auto-Regularisierung. Wird eine Liste angegeben, werden Modelle für jedes Verhältnis trainiert und das beste ausgewählt. | Nein | `0.75` |
| `regularization_upper_bound` | `float` oder `list` | Obere Schranke(n) für den Regularisierungsparameter bei Auto-Regularisierung. Wird eine Liste angegeben, werden Modelle für jede obere Schranke trainiert und das beste ausgewählt. | Nein | `200` |
| `weight_update_method` | `str` | Methode zur Aktualisierung der Stichprobengewichte aus `logarithmic` und `quadratic`. | Nein | `logarithmic` |
| `sample_scaling` | `boolean` | Ob Stichprobenskalierung angewendet werden soll. | Nein | `False` |
| `prediction_scaling` | `float` | Skalierungsfaktor für Vorhersagen. | Nein | `None` |
| `optimizer_options` | `dictionary` | QAOA-Optimierungsoptionen. Eine Liste der verfügbaren Optionen wird später in dieser Dokumentation aufgeführt. | Nein | ... |
| `voting` | `str` | Mehrheitsvoting (`hard`) oder Durchschnitt der Wahrscheinlichkeiten (`soft`) für die Aggregation der Vorhersagen/Wahrscheinlichkeiten der Lerner verwenden. | Nein | `hard` |
| `prob_threshold` | `float` | Optimaler Wahrscheinlichkeitsschwellenwert. | Nein | `0.5` |
| `random_state` | `integer` | Zufälligkeit für Reproduzierbarkeit steuern. | Nein | `None` |


- Zusätzlich werden die `optimizer_options` wie folgt aufgelistet:

|     Name    |    Typ      | Beschreibung | Erforderlich  | Standard |
|-------------|-------------|--------------|---------------|---------|
| `num_solutions` | `integer` | Die Anzahl der Lösungen | Nein | `1024` |
| `reps` | `integer` | Die Anzahl der Wiederholungen | Nein | `4` |
| `sparsify` | `float` | Der Sparsifikationsschwellenwert | Nein | `0.001` |
| `theta` | `float` | Der Anfangswert von Theta, ein Variationsparameter des QAOA | Nein | `None` |
| `simulator` | `boolean` | Ob ein Simulator oder ein QPU verwendet werden soll | Nein | `False` |
| `classical_optimizer` | `str` | Name des klassischen Optimierers für den QAOA. Alle von SciPy angebotenen Solver, wie [hier](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize) aufgelistet, sind verwendbar. Du musst `classical_optimizer_options` entsprechend setzen | Nein | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Optionen für den klassischen Optimierer. Eine vollständige Liste der verfügbaren Optionen findest du in der [SciPy-Dokumentation](https://docs.scipy.org/doc/scipy/reference/) | Nein | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Die Tiefe des QAOA-Circuit | Nein | `3` |
| `num_transpiler_runs` | `integer` | Anzahl der Transpiler-Läufe | Nein | `30` |
| `pass_manager_options` | `dictionary` | Optionen für den Preset-Pass-Manager | Nein | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Estimator-Optionen. Eine vollständige Liste der verfügbaren Optionen findest du in der [Qiskit Runtime Client-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Nein | `None` |
| `sampler_options` | `dictionary` | Sampler-Optionen. Eine vollständige Liste der verfügbaren Optionen findest du in der [Qiskit Runtime Client-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Nein | `None` |

- Standard-`estimator_options` sind:

|     Name    |    Typ      | Wert  |
|-------------|-------------|-------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Standard-`sampler_options` sind:

|     Name    |    Typ      | Wert |
|-------------|-------------|------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Verwendung

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Validierungen
- `name`:
    - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Es darf kein Klassifikator mit demselben Namen im gemeinsamen Datenverzeichnis vorhanden sein.
### 3. Delete
Die Aktion `delete` entfernt einen Klassifikator aus dem gemeinsamen Datenverzeichnis.

#### Eingaben
|     Name    |    Typ      | Beschreibung |   Erforderlich  |
|-------------|-------------|--------------|-----------------|
| `action`    | `str`    | Der Name der Aktion. Muss `delete` sein. | Ja |
| `name`      | `str`    | Der Name des zu löschenden Klassifikators. | Ja |

#### Verwendung

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Validierungen
- `name`:
    - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Ein Klassifikator mit demselben Namen muss bereits im gemeinsamen Datenverzeichnis vorhanden sein.
### 4. Fit
Die Aktion `fit` trainiert einen Klassifikator mit den bereitgestellten Trainingsdaten.

#### Eingaben
|     Name    |    Typ      | Beschreibung |   Erforderlich  |
|-------------|-------------|--------------|-----------------|
| `action`    | `str`    | Der Name der Aktion. Muss `fit` sein. | Ja |
| `name`      | `str`    | Der Name des zu trainierenden Klassifikators. | Ja |
| `X`         | `array` oder `list` oder `str` | Die Trainingsdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `y`         | `array` oder `list` oder `str` | Die Trainingszielwerte. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `fit_params`| `dictionary`| Zusätzliche Parameter, die an die Methode `fit` des Klassifikators übergeben werden. | Nein |

##### fit_params
|     Name    |    Typ      | Beschreibung |   Erforderlich  |   Standard   |
|-------------|-------------|--------------|-----------------|--------------|
| `validation_data` | `tuple` | Die Validierungsdaten und Labels. | Nein | `None` |
| `pos_label` | `integer` oder `str` | Das Klassenzeichen, das auf 1 abgebildet werden soll. | Nein | `None` |
| `optimization_data` | `str` | Datensatz zur Optimierung des Ensembles. Kann eines der folgenden sein: `train`, `validation`, `both`. | Nein | `train` |

#### Verwendung

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Validierungen
- `name`:
    - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Ein Klassifikator mit demselben Namen muss bereits im gemeinsamen Datenverzeichnis vorhanden sein.
### 5. Predict
Die Aktion `predict` wird verwendet, um harte und weiche Vorhersagen (Wahrscheinlichkeiten) zu erhalten.

#### Eingaben
|     Name    |    Typ      | Beschreibung |   Erforderlich  |
|-------------|-------------|--------------|-----------------|
| `action`    | `str`    | Der Name der Aktion. Muss `predict` sein. | Ja |
| `name`      | `str`    | Der Name des zu verwendenden Klassifikators. | Ja |
| `X`         | `array` oder `list` oder `str` | Die Testdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `options["out"]` | `str` | Der JSON-Ausgabedateiname zum Speichern der Vorhersagen im gemeinsamen Datenverzeichnis. Wenn nicht angegeben, werden die Vorhersagen im Job-Ergebnis zurückgegeben. | Nein |

#### Verwendung

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Validierungen
- `name`:
    - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Ein Klassifikator mit demselben Namen muss bereits im gemeinsamen Datenverzeichnis vorhanden sein.
- `options["out"]`:
    - Der Dateiname muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Er muss die Erweiterung `.json` haben.
### 6. Fit-predict
Die Aktion `fit_predict` trainiert einen Klassifikator mit den Trainingsdaten und verwendet ihn anschließend, um harte und weiche Vorhersagen (Wahrscheinlichkeiten) zu erhalten.

#### Eingaben
|     Name    |    Typ      | Beschreibung |   Erforderlich  |
|-------------|-------------|--------------|-----------------|
| `action`    | `str`    | Der Name der Aktion. Muss `fit_predict` sein. | Ja |
| `name`      | `str`    | Der Name des zu verwendenden Klassifikators. | Ja |
| `X_train`   | `array` oder `list` oder `str` | Die Trainingsdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `y_train`   | `array` oder `list` oder `str` | Die Trainingszielwerte. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `X_test`    | `array` oder `list` oder `str` | Die Testdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `fit_params`| `dictionary`| Zusätzliche Parameter, die an die Methode `fit` des Klassifikators übergeben werden. | Nein |
| `options["out"]` | `str` | Der JSON-Ausgabedateiname zum Speichern der Vorhersagen im gemeinsamen Datenverzeichnis. Wenn nicht angegeben, werden die Vorhersagen im Job-Ergebnis zurückgegeben. | Nein |

#### Verwendung

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Validierungen
- `name`:
    - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Ein Klassifikator mit demselben Namen muss bereits im gemeinsamen Datenverzeichnis vorhanden sein.

- `options["out"]`:
    - Der Dateiname muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Er muss die Erweiterung `.json` haben.
### 7. Create-fit-predict
Die Aktion `create_fit_predict` erstellt einen Klassifikator, trainiert ihn mit den bereitgestellten Trainingsdaten und verwendet ihn anschließend, um harte und weiche Vorhersagen (Wahrscheinlichkeiten) zu erhalten.

#### Eingaben
| Name | Typ | Beschreibung | Erforderlich |
|------|-----|--------------|-------------|
| `action` | `str` | Der Name der Aktion aus `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` und `delete`. | Ja |
| `name` | `str` | Der Name des zu verwendenden Klassifikators. | Ja |
| `quantum_classifier` | `str` | Der Typ des Klassifikators, d.h. `QuantumEnhancedEnsembleClassifier`. Standard ist `QuantumEnhancedEnsembleClassifier`. | Nein |
| `X_train` | `array` oder `list` oder `str` | Die Trainingsdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `y_train` | `array` oder `list` oder `str` | Die Trainingszielwerte. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `X_test` | `array` oder `list` oder `str` | Die Testdaten. Dies kann ein NumPy-Array, eine Liste oder eine Zeichenkette sein, die auf einen Dateinamen im gemeinsamen Datenverzeichnis verweist. | Ja |
| `fit_params` | `dictionary` | Zusätzliche Parameter, die an die Methode `fit` des Klassifikators übergeben werden. | Nein |
| `options["save"]` | `boolean` | Ob der trainierte Klassifikator im gemeinsamen Datenverzeichnis gespeichert werden soll. Standard ist `True`. | Nein |
| `options["out"]` | `str` | Der JSON-Ausgabedateiname zum Speichern der Vorhersagen im gemeinsamen Datenverzeichnis. Wenn nicht angegeben, werden die Vorhersagen im Job-Ergebnis zurückgegeben. | Nein |

#### Verwendung

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Validierungen
- `name`:
    - Wenn `options["save"]` auf `True` gesetzt ist:
        - Der Name muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
        - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
        - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
        - Es darf kein Klassifikator mit demselben Namen im gemeinsamen Datenverzeichnis vorhanden sein.

- `options["out"]`:
    - Der Dateiname muss eindeutig sein, eine Zeichenkette bis zu 64 Zeichen lang.
    - Er darf nur alphanumerische Zeichen und Unterstriche enthalten.
    - Er muss mit einem Buchstaben beginnen und darf nicht mit einem Unterstrich enden.
    - Er muss die Erweiterung `.json` haben.
---
## Erste Schritte
Authentifiziere dich mit deinem [IBM Quantum Platform API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle die Qiskit-Function wie folgt aus:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Beispiel
In diesem Beispiel verwendest du die Funktion „Singularity Machine Learning - Classification", um einen Datensatz zu klassifizieren, der aus zwei ineinandergreifenden, mondförmigen Halbkreisen besteht. Der Datensatz ist synthetisch, zweidimensional und mit binären Labels versehen. Er wurde so erstellt, dass er für Algorithmen wie zentroidbasiertes Clustering und lineare Klassifikation besonders herausfordernd ist.
![Moons-Datensatz](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
In diesem Prozess lernst du, wie du den Klassifikator erstellst, ihn an die Trainingsdaten anpasst, damit Vorhersagen auf den Testdaten triffst und den Klassifikator nach Abschluss löschst.
Bevor du beginnst, musst du [scikit-learn](https://scikit-learn.org/) installieren. Installiere es mit folgendem Befehl:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Führe die folgenden Schritte durch:

1) Erstelle den synthetischen Datensatz mit der Funktion [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) aus [scikit-learn](https://scikit-learn.org/).
2) Lade den erzeugten synthetischen Datensatz in das [gemeinsame Datenverzeichnis](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html) hoch.
3) Erstelle den quantengestützten Klassifikator mit der Aktion [create](#2-create).
4) Liste deine Klassifikatoren mit der Aktion [list](#1-list) auf.
5) Trainiere den Klassifikator auf den Trainingsdaten mit der Aktion [fit](#4-fit).
6) Verwende den trainierten Klassifikator, um mit der Aktion [predict](#5-predict) Vorhersagen auf den Testdaten zu treffen.
7) Lösche den Klassifikator mit der Aktion [delete](#3-delete).
8) Räume nach dem Abschluss auf.
**Schritt 1.** Importiere die erforderlichen Module und generiere den synthetischen Datensatz, dann teile ihn in Trainings- und Testdatensätze auf.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Schritt 2.** Speichere die gelabelten Trainings- und Testdatensätze auf deiner lokalen Festplatte und lade sie dann in das [gemeinsame Datenverzeichnis](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html) hoch.

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Schritt 3.** Erstelle einen quantengestützten Klassifikator mit der Aktion [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>